Notebook to plot various vertical profiles, time series, and histograms from WRF TC output for Haiyan and Maria.

Figs. 2, 3 of paper, plus multiple figures included in supporting information.

James Ruppert  
jruppert@ou.edu  
3/14/26

## Main settings

In [ ]:
import numpy as np
from matplotlib import ticker, rc
import matplotlib.pyplot as plt
from scipy import stats
from thermo_functions import *
import pickle
import seaborn as sns

In [ ]:
# #### Main settings

storm = 'haiyan'
# storm = 'maria'

data_main = "./data/"
figdir = "./figures/"

# Members
nmem = 10 # number of ensemble members (1-5 have NCRF)
enstag = str(nmem)

In [ ]:
# Ensemble member info
memb0=1 # Starting member to read
memb_nums=np.arange(memb0,nmem+memb0,1)
memb_nums_str=memb_nums.astype(str)
nustr = np.char.zfill(memb_nums_str, 2)
memb_all=np.char.add('memb_',nustr)

# Get dimensions
def get_nt(test_str):
    if test_str == 'ctl':
        nt = 97
    else:
        nt = 49
    nz = 39
    pres = np.arange(1000, 25, -25)
    return nt, nz, pres
nt, nz, pres = get_nt('ctl')
nz=39
dp = (pres[0]-pres[1])*1e2 # Pa

In [ ]:
# Tests to read and compare
if storm == 'haiyan':
    tests = ['ctl','ncrf36h','STRATANVIL_ON','STRATANVIL_OFF','STRAT_OFF']
    tests_str = ['CTL','Off-All','Off-Conv','Off-StratAnv','Off-Strat']
elif storm == 'maria':
    tests = ['ctl','ncrf48h']
    tests_str = ['CTL','Off-All']
ntest = len(tests)

# Baseline for sensitivity tests
if storm == 'haiyan':
    t0_tests=36
elif storm == 'maria':
    t0_tests=48
nt_test, nz, pres = get_nt(tests[1])
t1_tests = t0_tests + nt_test

# Precip/cloud classifications calculated using hydrometeor-based
# classification of Luschen and Ruppert (2024, GRL)
# 0-5: pclass 0-5
# 6: pclass=1,4,5 (deep conv + strat + anvil)
# 7 = where PW ≥ 48 (Mapes et al. 2018)
# 8 = where PW < 48 (Mapes et al. 2018)
# 9 = whole domain
nmean=10
mean_str = ["Non-cloud", "Deep", "Cong", "Shallow", "Strat", "Anvil", 'MCS', 'Moist', 'Dry', 'All']
mean_str_names = ["Non-cloud", "Deep", "Cong", "Shallow", "Strat", "Anvil", 'DSA', 'Moist', 'Dry', 'All']

# Subset/selection
imean_sel = [1,4,5,6]
mean_str = [mean_str[imean_sel[i]] for i in range(len(imean_sel))]
mean_str_names = [mean_str_names[imean_sel[i]] for i in range(len(imean_sel))]
nmean = len(imean_sel)

## Read variables

### Read main data

In [ ]:
# Time bounds for each test
all_t0=np.zeros(ntest, dtype=np.int8)
all_t1=np.zeros(ntest, dtype=np.int8)
for itest in range(ntest):
    test_str = tests[itest]
    nt_test, nz, pres = get_nt(test_str)
    if test_str == 'ctl':
        all_t0[itest]=0
    else:
        all_t0[itest]=t0_tests
    all_t1[itest]=all_t0[itest]+nt_test

In [ ]:
# Main read loops

# Arrays to save variables

# 3D Variables
dims = (nmem, ntest, nmean, nt, nz)
w     = np.full(dims, np.nan)
rho   = np.full(dims, np.nan)
qv    = np.full(dims, np.nan)
tmpk  = np.full(dims, np.nan)
lw    = np.full(dims, np.nan)
lwc   = np.full(dims, np.nan)
sw    = np.full(dims, np.nan)
swc   = np.full(dims, np.nan)
z     = np.full(dims, np.nan)
condh = np.full(dims, np.nan)
dse_u_adv = np.full(dims, np.nan)
dse_v_adv = np.full(dims, np.nan)

# 2D Variables
dims = (nmem, ntest, nmean, nt)
mse_vint      = np.full(dims, np.nan)
vadv_mse_vint = np.full(dims, np.nan)
vadv_dse_vint = np.full(dims, np.nan)
vmf           = np.full(dims, np.nan)
vmfu          = np.full(dims, np.nan)
vmfd          = np.full(dims, np.nan)
rain          = np.full(dims, np.nan)
condh_kuk     = np.full(dims, np.nan)

# Loop over ensemble members
for imemb in range(nmem):

    for itest in range(ntest):

        test_str = tests[itest]
        t0 = all_t0[itest]
        t1 = all_t1[itest]

        pickle_file = f"{data_main}mean_profiles_{test_str}_{storm}_{memb_all[imemb]}.pkl"
        with open(pickle_file, 'rb') as file:
            allvars_mean = pickle.load(file)

        for imean in range(nmean):
            # 3D Variables
            w[imemb,itest,imean,t0:t1,...]     = allvars_mean['W'][mean_str[imean]] # m/s
            rho[imemb,itest,imean,t0:t1,...]   = allvars_mean['rho'][mean_str[imean]] # kg/m3
            qv[imemb,itest,imean,t0:t1,...]    = allvars_mean['QVAPOR'][mean_str[imean]] # kg/kg
            tmpk[imemb,itest,imean,t0:t1,...]  = allvars_mean['T'][mean_str[imean]] # K
            lw[imemb,itest,imean,t0:t1,...]    = allvars_mean['RTHRATLW'][mean_str[imean]] # K/s
            lwc[imemb,itest,imean,t0:t1,...]   = allvars_mean['RTHRATLWC'][mean_str[imean]] # K/s
            sw[imemb,itest,imean,t0:t1,...]    = allvars_mean['RTHRATSW'][mean_str[imean]] # K/s
            swc[imemb,itest,imean,t0:t1,...]   = allvars_mean['RTHRATSWC'][mean_str[imean]] # K/s
            z[imemb,itest,imean,t0:t1,...]     = allvars_mean['Z'][mean_str[imean]] # m
            condh[imemb,itest,imean,t0:t1,...] = allvars_mean['condh'][mean_str[imean]] # J/kg/s
            # 2D Variables
            vmf[imemb,itest,imean,t0:t1]           = allvars_mean['vmf'][mean_str[imean]] # kg/m/s
            vmfu[imemb,itest,imean,t0:t1]          = allvars_mean['vmfu'][mean_str[imean]] # kg/m/s
            vmfd[imemb,itest,imean,t0:t1]          = allvars_mean['vmfd'][mean_str[imean]] # kg/m/s
            rain[imemb,itest,imean,t0:t1]          = allvars_mean['rain'][mean_str[imean]] # mm/(time step)
            condh_kuk[imemb,itest,imean,t0:t1,...] = allvars_mean['condh_kukulies'][mean_str[imean]] # kg/m2/s

# Set first time step to NaN for all tests
rain[:,:,:,(t0_tests,t1_tests-1)] = np.nan

# Different rain units
lv0=2.5e6 # J/kg
rain_kgpms = rain/3600 # mm/(time step) --> mm/s = kg/m2/s
# J/kg * kg/m2/s = W/m2

### Read pclass

In [ ]:
# Main read loops for 3D (dependent) variables
pclass_names=["Non-cloud", "Deep", "Cong", "Shallow", "Stratiform", "Anvil", "MCS"]
npclass=len(pclass_names)

# Arrays to save variables
dims_area = (nmem, ntest, npclass, nt)
pclass_area = np.full(dims_area, np.nan)

for itest in range(ntest):
    test_str=tests_str[itest]
    test_read=tests[itest]
    print('Reading test: ',test_str)
    # Loop over ensemble members
    for imemb in range(nmem):
        pickle_file = f"{data_main}pclass_{test_read}_48hrs_{storm}_{memb_all[imemb]}.pkl"
        with open(pickle_file, 'rb') as file:
            pclass_area[imemb,itest,:,t0_tests+1:t0_tests+nt_test] = pickle.load(file)

### Read pclass clusters

In [ ]:
# Arrays to save variables
dims = (ntest, nmem, npclass, nt)
cluster_number = np.full(dims, np.nan)
cluster_size = np.full(dims, np.nan)

for itest in range(ntest):
    test_read=tests[itest]
    for imemb in range(nmem):
        pickle_cluster = f"{data_main}pclass_cluster_{test_read}_48hrs_{storm}_{memb_all[imemb]}.pkl"
        with open(pickle_cluster, 'rb') as file:
            inumber, isize = pickle.load(file)
        cluster_number[itest,imemb,:,t0_tests+1:t0_tests+nt_test]    = inumber
        cluster_size[itest,imemb,:,t0_tests+1:t0_tests+nt_test] = isize

### New variables

In [ ]:
# Vertical integral function
def calc_vint(invar, pres, dp):
    # Note: pres is in hPa, dp in Pa
    g = 9.81 # m/s2
    p_top = 100 # hPa, integration top
    ik_sum = np.where(pres >= p_top)
    var_vint = np.sum(invar[...,0:np.max(ik_sum)+1], axis=-1)*dp/g
    return var_vint

In [ ]:
# Vertical integral function
def calc_vavg(invar, pres, dp):
    # Note: pres is in hPa, dp in Pa
    p_top = 100 # hPa, integration top
    ik_sum = np.where(pres >= p_top)
    # Don't need dp, since dp is constant and it divides out for average
    var_vavg = np.mean(invar[...,0:np.max(ik_sum)+1], axis=-1)#*dp
    return var_vavg

In [ ]:
plevs_full = (1100, 100)
plevs_low = (1100, 700)
plevs_mid = (750, 400)
plevs_high = (400, 100)
subtitles = ['Column', 'Low', 'Mid', 'High']

# Vertical integral function
def calc_vint_layers(invar, pres, dp, p_bot, p_top):
    # Note: pres is in hPa, dp in Pa
    g = 9.81 # m/s2
    # p_top = 100 # hPa, integration top
    ik_sum = np.where((pres < p_bot) & (pres >= p_top))
    var_vint = np.sum(invar[...,np.min(ik_sum):np.max(ik_sum)+1], axis=-1)*dp/g
    return var_vint

# Vertical integral function
def calc_vavg_layers(invar, pres, dp, p_bot, p_top):
    # Note: pres is in hPa, dp in Pa
    ik_sum = np.where((pres < p_bot) & (pres >= p_top))
    var_vint = np.mean(invar[...,np.min(ik_sum):np.max(ik_sum)+1], axis=-1)
    return var_vint

#### PW_Sat saturation CRH

In [ ]:
# Calculate PW and PW_SAT for saturation fraction
qv_sat = rv_saturation(tmpk, 
                       (pres[np.newaxis,np.newaxis,np.newaxis,np.newaxis,:])*1e2) # kg/kg
# Integrate over full column
pw = calc_vint(qv, pres, dp) # mm
pw_sat = calc_vint(qv_sat, pres, dp) # mm
crh = pw/pw_sat

#### Theta, VMF

In [ ]:
thv = theta_virtual(tmpk, qv, pres[np.newaxis, np.newaxis, np.newaxis, np.newaxis, :]*1e2) # K
the = theta_equiv(tmpk, qv, qv, pres[np.newaxis, np.newaxis, np.newaxis, np.newaxis, :]*1e2) # K
the = np.ma.filled(the, np.nan)
relh = calc_relh(qv, pres[np.newaxis, np.newaxis, np.newaxis, np.newaxis, :]*1e2, tmpk, ice=True) # %

lwcrf = (lw - lwc) * (24*3600) # K/s --> K/d
swcrf = (sw - swc) * (24*3600) # K/s --> K/d
vmf_prof = w*rho # kg/m3 m/s --> kg/m2/s
qv_prof = qv*1e3 # kg/kg --> g/kg
# vmf_prof = w # m/s

# Vertically integrated condh
condh_vint = calc_vint(condh, pres, dp) # J/kg/s --> W/m2

# MSE
# mse = calc_mse(tmpk, qv, z+zb_prof) # J/kg

# Vertical integrals/averages
thv_vavg = calc_vavg(thv, pres, dp) # K
the_vavg = calc_vavg(the, pres, dp) # K
the_pbl = calc_vavg_layers(the, pres, dp, 1100, 850) # K
thv_pbl = calc_vavg_layers(thv, pres, dp, 1100, 850) # K
the_freetrop = calc_vavg_layers(the, pres, dp, 700, 500) # K
thv_freetrop = calc_vavg_layers(thv, pres, dp, 700, 500) # K
# the_diff = the_freetrop - the_pbl
the_diff = the_pbl - the_freetrop
# thv_diff = thv_freetrop - thv_pbl
thv_diff = thv_pbl - thv_freetrop

# CRH in the PBL
pw_pbl = calc_vint_layers(qv, pres, dp, 1100, 850) # mm
pw_sat_pbl = calc_vint_layers(qv_sat, pres, dp, 1100, 850) # mm
crh_pbl = pw_pbl/pw_sat_pbl

cp=1004 # J/K/kg
lwcrf_vint = calc_vint(lwcrf*cp/(24*3600), pres, dp) # --> W/m2
swcrf_vint = calc_vint(swcrf*cp/(24*3600), pres, dp) # --> W/m2

condh_kpd = condh * (24*3600)/cp # J/kg/s --> K/s --> K/d

#### WTG

##### Equations:
Full equation:
$$\dfrac{d\overline s}{dt} = \dfrac{\partial\overline s}{\partial t} + \mathbf{\overline v} \cdot \nabla\overline s + \overline w \dfrac{d\overline s}{dz} = Q_1$$

WTG balance:
$$w\dfrac{d\overline s}{dz} = Q_1$$

WTG assessment:
$$\alpha_{ddt} = \left( \dfrac{\partial\overline s}{\partial t} \right) / \left( \overline w \dfrac{d\overline s}{dz} \right)$$
$$\alpha_{ddt} = \left( \mathbf{\overline v} \cdot \nabla\overline s \right) / \left( \overline w \dfrac{d\overline s}{dz} \right)$$

WTG diagnostics:
$$w_{wtg} = Q_1/\dfrac{d\overline s}{dz}$$
$$w_{CRF} = Q_{CRF}/\dfrac{d\overline s}{dz}$$
$$w\dfrac{d\overline q}{dz} = -Q_2$$
$$\dot q_{CRF} = - w_{CRF} \dfrac{d\overline q}{dz} = - Q_{CRF} \dfrac{d\overline q}{dz} /\dfrac{d\overline s}{dz}$$

In [ ]:
# Dry static energy (DSE)

# Need base state z, since was not added to z write-out
# datdir = '../'
# zb = var_read_zb_hires(datdir, mask=True, drop=True)
# zb_prof=zb[0,:,500,500]
# z_prof = z + zb_prof[np.newaxis,np.newaxis,np.newaxis,np.newaxis,:]

# Constants
cp=1004.  # J/K/kg
g=9.81 # m/s2
dse = cp*tmpk + g*z # J/kg

# Vertical DSE gradient
dims = (nmem, ntest, nmean, nt, nz)
dsdz = np.full(dims, np.nan) # J/kg/m
dqdz = np.full(dims, np.nan) # kg/kg/m

for imemb in range(nmem):
    for itest in range(ntest):
        for imean in range(nmean):
            for it in range(nt):
                dsdz[imemb, itest, imean, it, :] = np.gradient(dse[imemb, itest, imean, it, :], z[imemb, itest, imean, it, :])
                dqdz[imemb, itest, imean, it, :] = np.gradient(qv[imemb, itest, imean, it, :], z[imemb, itest, imean, it, :])

In [ ]:
# WTG assessment terms
nt_p_timestep = 3600 # s (1 hour)
# dse_ttend = np.gradient(dse, axis=3) / nt_p_timestep # J/kg/s
dse_ttend = np.full(dims, np.nan) # J/kg/s
for itest in range(ntest):
    dse_ttend[:,itest,:,all_t0[itest]:all_t1[itest], :] = \
        np.gradient(dse[:,itest,:,all_t0[itest]:all_t1[itest],:], axis=2) / nt_p_timestep # J/kg/s
dse_hor_adv = dse_u_adv + dse_v_adv # J/kg/s
dse_w_adv = w * dsdz # J/kg/s

# alpha_ddt = dse_ttend / dse_w_adv
# alpha_hadv = dse_hor_adv / dse_w_adv
# alpha_wtg = (dse_ttend + dse_hor_adv) / dse_w_adv

##### WTG CRF

In [ ]:
# Q
dse_w_adv_lw = cp*lw # J/kg/s
dse_w_adv_sw = cp*sw # J/kg/s
dse_w_adv_tot = dse_w_adv_lw + dse_w_adv_sw

dse_w_adv_lwcrf = cp*(lw - lwc) # J/kg/s
dse_w_adv_swcrf = cp*(sw - swc) # J/kg/s
dse_w_adv_totcrf = dse_w_adv_lwcrf + dse_w_adv_swcrf

# WTG
w_wtg_lw = dse_w_adv_lw / dsdz # J/kg/s / J/kg/m --> m/s
w_wtg_sw = dse_w_adv_sw / dsdz # J/kg/s / J/kg/m --> m/s
w_wtg_tot = dse_w_adv_tot / dsdz # J/kg/s / J/kg/m --> m/s
w_wtg_lwcrf = dse_w_adv_lwcrf / dsdz # J/kg/s / J/kg/m --> m/s
w_wtg_swcrf = dse_w_adv_swcrf / dsdz # J/kg/s / J/kg/m --> m/s
w_wtg_totcrf = dse_w_adv_totcrf / dsdz # J/kg/s / J/kg/m --> m/s

# Vertical integrals of WTG terms
w_wtg_lw_vint = calc_vint(w_wtg_lw, pres, dp) # m/s --> m/s . s2/m . kg/m/s2 --> kg/m/s
w_wtg_sw_vint = calc_vint(w_wtg_sw, pres, dp) # m/s --> m/s . s2/m . kg/m/s2 --> kg/m/s
w_wtg_tot_vint = calc_vint(w_wtg_tot, pres, dp) # m/s --> m/s . s2/m . kg/m/s2 --> kg/m/s
w_wtg_lwcrf_vint = calc_vint(w_wtg_lwcrf, pres, dp) # m/s --> m/s . s2/m . kg/m/s2 --> kg/m/s
w_wtg_swcrf_vint = calc_vint(w_wtg_swcrf, pres, dp) # m/s --> m/s . s2/m . kg/m/s2 --> kg/m/s
w_wtg_totcrf_vint = calc_vint(w_wtg_totcrf, pres, dp) # m/s --> m/s . s2/m . kg/m/s2 --> kg/m/s

# Convert profile to VMF units
w_wtg_lw *= rho # m/s --> kg/m2/s
w_wtg_sw *= rho # m/s --> kg/m2/s
w_wtg_tot *= rho # m/s --> kg/m2/s
w_wtg_lwcrf *= rho # m/s --> kg/m2/s
w_wtg_swcrf *= rho # m/s --> kg/m2/s
w_wtg_totcrf *= rho # m/s --> kg/m2/s

# Delta from CTL
w_wtg_lwcrf_delta = w_wtg_lwcrf - (w_wtg_lwcrf[:,0,...])[:,np.newaxis,...]
w_wtg_totcrf_delta = w_wtg_totcrf - (w_wtg_totcrf[:,0,...])[:,np.newaxis,...]
dse_w_adv_lwcrf_delta = dse_w_adv_lwcrf - (dse_w_adv_lwcrf[:,0,...])[:,np.newaxis,...]

##### WTG LH

In [ ]:
w_wtg_condh = condh / dsdz # J/kg/s / J/kg/m --> m/s
w_wtg_condh_vint = calc_vint(w_wtg_condh, pres, dp) # m/s --> m/s . s2/m . kg/m/s2 --> kg/m/s
# Convert profile to VMF units
w_wtg_condh *= rho # m/s --> kg/m2/s
w_wtg_condh_delta = w_wtg_condh - (w_wtg_condh[:,0,...])[:,np.newaxis,...]

##### WTG SUM

In [ ]:
# Already in VMF units
w_wtg_sum_delta = w_wtg_lwcrf_delta + w_wtg_condh_delta

##### VADV moisture, WTG

In [ ]:
lv = 2.5e6 # J/kg

# Actual vertical moisture advection
dqdt_vadv = - w * dqdz # m/s * kg/kg / m --> /s
# Vertical integral:
dqdt_vadv_vint = calc_vint(dqdt_vadv*lv, pres, dp) # J/kg/s --> W/kg . s2/m . kg/m/s2 --> W/m2

# WTG from Radiation terms
dqdt_vadv_lw = - w_wtg_lw * dqdz/rho # m/s * kg/kg / m --> /s
dqdt_vadv_sw = - w_wtg_sw * dqdz/rho # m/s * kg/kg / m --> /s
dqdt_vadv_tot = - w_wtg_tot * dqdz/rho # m/s
dqdt_vadv_lwcrf = - w_wtg_lwcrf * dqdz/rho # m/s * kg/kg / m --> /s
dqdt_vadv_swcrf = - w_wtg_swcrf * dqdz/rho # m/s * kg/kg / m --> /s
dqdt_vadv_totcrf = - w_wtg_totcrf * dqdz/rho # m/s * kg/kg / m --> /s
# Vertical integral:
dqdt_vadv_lw_vint = calc_vint(dqdt_vadv_lw*lv, pres, dp) # J/kg/s --> W/kg . s2/m . kg/m/s2 --> W/m2
dqdt_vadv_sw_vint = calc_vint(dqdt_vadv_sw*lv, pres, dp) # J/kg/s --> W/kg . s2/m . kg/m/s2 --> W/m2
dqdt_vadv_tot_vint = calc_vint(dqdt_vadv_tot*lv, pres, dp) # J/kg/s --> W/kg . s2/m . kg/m/s2 --> W/m2
dqdt_vadv_lwcrf_vint = calc_vint(dqdt_vadv_lwcrf*lv, pres, dp) # J/kg/s --> W/kg . s2/m . kg/m/s2 --> W/m2
dqdt_vadv_swcrf_vint = calc_vint(dqdt_vadv_swcrf*lv, pres, dp) # J/kg/s --> W/kg . s2/m . kg/m/s2 --> W/m2
dqdt_vadv_totcrf_vint = calc_vint(dqdt_vadv_totcrf*lv, pres, dp) # J/kg/s --> W/kg . s2/m . kg/m/s2 --> W/m2

# WTG from condensational heating
dqdt_vadv_condh = - w_wtg_condh * dqdz/rho # m/s * kg/kg / m --> /s
# Vertical integral:
dqdt_vadv_condh_vint = calc_vint(dqdt_vadv_condh*lv, pres, dp) # J/kg/s --> W/kg . s2/m . kg/m/s2 --> W/m2

In [ ]:
# Convert to water tendency
dqdt_vadv_vint /= lv # W/m2 --> kg/m2/s = mm/s of rain equivalent
dqdt_vadv_lwcrf_vint /= lv # W/m2 --> kg/m2/s = mm/s of rain equivalent
dqdt_vadv_condh_vint /= lv # W/m2 --> kg/m2/s = mm/s of rain equivalent

# Convert to mm/d
tommpd = 24*3600 # s/d
dqdt_vadv_vint *= tommpd # mm/s --> mm/d
dqdt_vadv_lwcrf_vint *= tommpd # mm/s --> mm/d
dqdt_vadv_condh_vint *= tommpd # mm/s --> mm/d

In [ ]:
# Free trop integrals
# plevs_int_mid = [900, 400]
plevs_int_mid = [800, 400]
# plevs_int_mid = [700, 300]
# plevs_int = [1200, 100]
pw_mid = calc_vint_layers(qv, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # kg/m2 = mm of precip equivalent
pw_sat_mid = calc_vint_layers(qv_sat, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # mm
crh_mid = pw_mid/pw_sat_mid

dqdt_actual = np.gradient(pw_mid, axis=-1)*24 # mm/hr --> mm/d

dqdt_vadv_mid = calc_vint_layers(dqdt_vadv, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # mm/s of rain equivalent

dqdt_vadv_lw_mid = calc_vint_layers(dqdt_vadv_lw, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # J/kg/s --> W/m2 --> mm/s of rain equivalent
dqdt_vadv_sw_mid = calc_vint_layers(dqdt_vadv_sw, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # J/kg/s --> W/m2 --> mm/s of rain equivalent
dqdt_vadv_lwcrf_mid = calc_vint_layers(dqdt_vadv_lwcrf, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # mm/s of rain equivalent
dqdt_vadv_swcrf_mid = calc_vint_layers(dqdt_vadv_swcrf, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # mm/s of rain equivalent
dqdt_vadv_condh_mid = calc_vint_layers(dqdt_vadv_condh, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # mm/s of rain equivalent

# DSE WTG terms
dse_ttend_vint_mid = calc_vint_layers(dse_ttend, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # J/kg/s --> W/m2
dse_vadv_vint_mid = calc_vint_layers(dse_w_adv, pres, dp, plevs_int_mid[0], plevs_int_mid[1]) # J/kg/s --> W/m2

# Convert to mm/d
dqdt_vadv_mid *= tommpd # mm/s --> mm/d
dqdt_vadv_lw_mid *= tommpd # mm/s --> mm/d
dqdt_vadv_sw_mid *= tommpd # mm/s --> mm/d
dqdt_vadv_lwcrf_mid *= tommpd # mm/s --> mm/d
dqdt_vadv_swcrf_mid *= tommpd # mm/s --> mm/d
dqdt_vadv_condh_mid *= tommpd # mm/s --> mm/d

##### Chikira parameter

In [ ]:
chikira = - dqdz/dsdz # dq/dz / ds/dz = dq/ds [kg/kg / J/kg = kg/J]

## Plotting stuff

### Plot functions

#### General plot settings

In [ ]:
font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 12}

rc('font', **font)

sns.set_theme(style="ticks", font_scale=1.2, rc={'xtick.bottom': True, 'ytick.left': True,
                                 "axes.spines.right": False, "axes.spines.top": False,})

In [ ]:
# Color and line settings
linecolor=['black', 'blue', 'red', 'green', 'green']
linestyle=['solid']*ntest # this creates an array of ['solid'] of size ntest
linestyle[-1]='dashed'

In [ ]:
# Confidence interval using T-test and assuming 95% significance
def mean_confidence_interval(data):
    # conf_set=0.9 # Confidence interval to apply throughout
    conf_set=0.95 # Confidence interval to apply throughout
    a = 1.0 * np.array(data)
    # n = len(a)
    n = a.shape[0]
    m, se = np.mean(a, axis=0), stats.sem(a, axis=0)
    # num = stats.t.ppf((1 + conf_set) / 2., n-1)
    h = se * stats.t.ppf((1 + conf_set) / 2., n-1)
    return m, m-h, m+h

In [ ]:
# Smooth time series
# Assumes f = f(nmem, nt), smooths only over 2nd dimension
# Simple 3-point moving average, with edge handling by keeping original value at edges
def smooth_tser(invar):
    invar_sm = np.copy(invar)
    invar_sm[:,1:-1] = (invar_sm[:,2:] + invar_sm[:,1:-1] + invar_sm[:,0:-2])/3
    return invar_sm

#### Vertical profile function

In [ ]:
def plot_profiles(title, all_vars_imean, figname=None, subtitles=None, units=None, ctlanom=False,
                  conf_shading=True, do_legend=False, wtg=None, xlim=None, do_ylog=True):

    nprof = all_vars_imean.shape[0]

    fig, ax = plt.subplots(1,nprof, figsize=(11,5), layout="constrained", dpi=300) # row, column
    # fig, ax = plt.subplots(1,nprof, figsize=(6,5), layout="constrained", dpi=300) # row, column

    # fig.suptitle(title)
    fig.supylabel('Pressure [hPa]')

    for iprof in range(nprof):

        # Compute means and time tendencies

        pltvar = all_vars_imean[iprof]
        pltvar_tavg = np.mean(pltvar, axis=2)

        # Difference from CTL
        # Calculate difference FOR EACH ENS MEMBER
        if ctlanom:
            pltvar_tavg -= pltvar_tavg[:,0,:][:,np.newaxis,:]

        # Plot all sensitivity tests for variable
        for itest in range(ntest):

            # Basic means with confidence intervals
            mean, low, high = mean_confidence_interval(pltvar_tavg[:,itest])
            label = tests_str[itest]
            if ctlanom and itest == 0:
                label = '_nolabel_'
            ax[iprof].plot(mean, pres, linestyle=linestyle[itest], color=linecolor[itest], label=label)
            if conf_shading:
                # if (itest == 0) or (itest == 1):
                ax[iprof].fill_betweenx(pres, low, high, alpha=0.2, color=linecolor[itest])

        ax[iprof].set_title(subtitles[iprof])#, fontsize=22)
        ax[iprof].set_xlabel(units[iprof])
        ax[iprof].invert_yaxis()
        if do_ylog:
            ax[iprof].set_yscale('log')
        ax[iprof].set_ylim(1000, 100)
        # if iprof == 0:
        ax[iprof].yaxis.set_major_formatter(ticker.ScalarFormatter())
        ax[iprof].yaxis.set_minor_formatter(ticker.ScalarFormatter())
        if xlim is not None:
            ax[iprof].set_xlim(xlim[iprof])
        ax[iprof].axvline(x=0,color='k',linewidth=0.5)

        sns.despine(offset=10,ax=ax[iprof])

    # WTG profile
    # if subtitles[iprof] == 'VMF' and wtg is not None:
    if wtg is not None:
        pltvar_wtg = np.mean(wtg, axis=2)
        # for itest in range(1,2):
        for itest in range(ntest):
            mean, low, high = mean_confidence_interval(pltvar_wtg[:,itest])
            ax[-1].plot(mean, pres, linestyle=':', linewidth=2, color=linecolor[itest])#, label=tests_str[itest]+'-WTG')
            if conf_shading:
                if (itest == 0) or (itest == 1):
                    ax[-1].fill_betweenx(pres, low, high, alpha=0.2, color=linecolor[itest])

    if do_legend:
        ax[-2].legend(loc="lower right", frameon=False, fontsize=10)

    if figname is not None:
        plt.savefig(figdir+figname+'.png',dpi=400, facecolor='white', \
                bbox_inches='tight')#, pad_inches=0.2)
    plt.show()
    return None

#### Time series functions

In [ ]:
def plot_time_series(title, all_vars_imean, figname=None, subtitles=None, units=None, ctlanom=False,
                     conf_shading=True, do_legend=True, xlim=None, ylim=None):
    nprof = all_vars_imean.shape[0]
    i_nt = all_vars_imean.shape[-1]
    # create figure
    fig_x = 7
    fig_y = 2.0*nprof + 1
    fig, axs = plt.subplots(nprof,1, figsize=(fig_x,fig_y), layout="constrained", dpi=300) # row, column
    # title = storm.capitalize()
    fig.suptitle(title)
    for iprof, ax in enumerate(axs):
        pltvar = np.copy(all_vars_imean[iprof])
        if ctlanom:
            pltvar -= pltvar[:,0,:][:,np.newaxis,:]
        # Plot all sensitivity tests for variable
        for itest in range(ntest):
            tser = pltvar[:,itest,:]
            # Smooth time series
            tser = smooth_tser(tser)
            mean, low, high = mean_confidence_interval(tser)
            ax.plot(mean, linestyle=linestyle[itest], color=linecolor[itest], label=tests_str[itest])
            if conf_shading:
                # if (itest == 0) or (itest == 1):
                xdim = range(0,i_nt)
                ax.fill_between(xdim, high, low, alpha=0.2, color=linecolor[itest])
        ax.set_title(subtitles[iprof])
        ax.set_ylabel(units[iprof])
        if iprof < nprof-1:
            sns.despine(offset=10,ax=ax, bottom=True)
            # Remove tick labels
            ax.set_xticks([])
        else:
            sns.despine(offset=10,ax=ax)
            ax.set_xlabel('Time [hours]')
        if xlim is not None:
            ax.set_xlim(xlim)
        if ylim is not None:
            ax.set_ylim(ylim[iprof])
        if do_legend and iprof == 0:
            ax.legend(loc="lower left", frameon=False, fontsize=9)#, bbox_to_anchor=(0.05, 0.05))
    if figname is not None:
        plt.savefig(figdir+'time_series_'+figname+'.png',dpi=400, facecolor='white', \
            bbox_inches='tight')#, pad_inches=0.2)
    plt.show()

In [ ]:
def plot_time_series_4x4(title, all_vars_imean, subtitles=None, units=None, ctlanom=False,
                     conf_shading=True, do_legend=True, do_titles=True):
    i_nt = all_vars_imean.shape[-1]
    # create figure
    fig, axs = plt.subplots(2,2, figsize=(13,5), layout="constrained", dpi=300) # row, column
    if do_titles:
        fig.suptitle(title)
    for iprof, ax in enumerate(axs.flat):
        pltvar = np.copy(all_vars_imean[iprof])
        if ctlanom:
            pltvar -= pltvar[:,0,:][:,np.newaxis,:]
        # Plot all sensitivity tests for variable
        for itest in range(ntest):
            tser = pltvar[:,itest,:]
            # Smooth time series
            tser = smooth_tser(tser)
            mean, low, high = mean_confidence_interval(tser)
            ax.plot(mean, linestyle=linestyle[itest], color=linecolor[itest], label=tests_str[itest])
            if conf_shading:
                # if (itest == 0) or (itest == 1):
                xdim = range(0,i_nt)
                ax.fill_between(xdim, high, low, alpha=0.2, color=linecolor[itest])
        ax.set_title(subtitles[iprof])
        ax.set_ylabel(units[iprof])
        # if iprof % 2 == 0:
        if iprof < 2:
            sns.despine(offset=10,ax=ax, bottom=True)
            # Remove tick labels
            ax.set_xticks([])
        else:
            sns.despine(offset=10,ax=ax)
            ax.set_xlabel('Time [hours]')
        if do_legend and iprof == 3:
            ax.legend(frameon=False, fontsize=12, bbox_to_anchor=(1.0, 1.3))
    plt.show()

In [ ]:
def run_tser_driver(all_vars, subtitles, units, figname=None, ctlanom=False, subtract_all=False,
                    nmean_plt=None, do_legend=True, fourxfour=False, do_titles=True, xlim=None,
                    ylim=None):

    # Plot only sensitivity test times
    all_vars = all_vars[...,t0_tests:t1_tests]

    # Subtract ALL-MEAN
    if subtract_all:
        all_vars -= all_vars[...,-1,:][...,np.newaxis,:]

    # Select means
    if nmean_plt is None:
        nmean_plt = nmean

    for imean in range(nmean_plt):
        # imean = imean_sel[iimean]
        if subtract_all:
            title = mean_str_names[imean]+'-ALL'
        else:
            title = mean_str_names[imean]
        if ctlanom:
            title += ', TEST-CTL'
        all_vars_imean = all_vars[...,imean,:]

        figname_pass = None
        if figname is not None:
            figname_pass = figname+'_'+mean_str_names[imean]+'_'+storm

        if fourxfour:
            plot_time_series_4x4(title, all_vars_imean, subtitles=subtitles, units=units, ctlanom=ctlanom,
                             conf_shading=True, do_legend=do_legend, do_titles=do_titles)
        else:
            plot_time_series(title, all_vars_imean, figname=figname_pass, subtitles=subtitles, units=units, ctlanom=ctlanom,
                             conf_shading=True, do_legend=do_legend, xlim=xlim, ylim=ylim)
    return None

#### Box plot function

In [ ]:
def boxplot_imean_tests(pltvar, labels, title_tag, units, yscale='linear', ctlanom=True):

    if ctlanom:
        cmap = ['teal', 'plum', 'darkorange', 'gold', 'cornflowerblue']
    else:
        cmap = ['gray', 'teal', 'plum', 'darkorange', 'gold', 'cornflowerblue']
    sns.set_palette(cmap)

    fig, ax = plt.subplots(1,1, figsize=(5,3), layout="constrained")#, dpi=300)
    nbins = len(labels)
    sns.boxplot([pltvar[:,ic] for ic in range(nbins)],
                width=0.6, showmeans=True, showfliers=False,
                meanprops={"marker":"o", "markerfacecolor":"white", 
                           "markeredgecolor":"black", "markersize":"6"},
                showcaps=False)

    plt.axhline(y=0, color='k', linestyle='-', linewidth=0.7, zorder=1)
    ax.set_yscale(yscale)
    # ax.set_ylim([1e-2,1e14])
    ax.set_xticklabels(labels)
    sns.despine(offset=10,ax=ax)
    plt.ylabel(units)#, weight='bold')
    plt.title(title_tag)
    plt.show()
    return None

### Plots

#### Vertical profiles

##### Basic profiles

In [ ]:
subtract_all = True
subtract_all = False

ctlanom = True
# ctlanom = False

wtg = True
wtg = False

# Plot variable

imean_mcs = 3
imean = imean_mcs

it_max_list = [6, 49]
nrows = len(it_max_list)

all_vars_base = np.array([lwcrf,
                     thv,
                     the,
                     relh,
                     vmf_prof*1e3,
                     ])

nprof = all_vars_base.shape[0]
subtitles = ['(a) LW-CRF',
             r'(b) $\theta_v$',
             r'(c) $\theta_e$',
             '(d) RH',
             '(e) VMF',
]
units     = ['K/d',
             'K',
             'K',
             '%',
             '10$^{-3}$ kg/m$^2$/s',
             ]

set_xlim = [
    (-3,5),
    (-0.3,0.35),
    (-0.3,0.35),
    (-1.5,2.3),
    (-9,2),
    ]

fig, axs = plt.subplots(nrows, nprof, figsize=(11, 5*nrows), layout="constrained", dpi=300)
fig.supylabel('Pressure [hPa]')

for irow, it_max in enumerate(it_max_list):

    all_vars = all_vars_base[...,t0_tests:t0_tests+it_max,:].copy()

    # Subtract ALL-MEAN
    if subtract_all:
        all_vars[1,...] -= all_vars[...,-1,:,:][1,...,np.newaxis,:,:]
        all_vars[2,...] -= all_vars[...,-1,:,:][2,...,np.newaxis,:,:]
        subtitles[2] = r"$\theta_e'$"

    all_vars_imean = all_vars[...,imean,:,:]

    if ctlanom:
        title = mean_str[imean]+', TEST-CTL '+r' ($\Delta t$='+str(it_max-1)+' hrs)'
    else:
        title = mean_str[imean]+r' ($\Delta t$='+str(it_max-1)+' hrs)'

    if wtg:
        wtg_prof = w_wtg_lwcrf_delta[...,imean,t0_tests:t0_tests+it_max,:]*1e3
    else:
        wtg_prof=None

    # Plot profiles into the corresponding row
    ax_row = axs[irow]

    for iprof in range(nprof):

        pltvar = all_vars_imean[iprof]
        pltvar_tavg = np.mean(pltvar, axis=2)

        if ctlanom:
            pltvar_tavg -= pltvar_tavg[:,0,:][:,np.newaxis,:]

        for itest in range(ntest):
            mean, low, high = mean_confidence_interval(pltvar_tavg[:,itest])
            label = tests_str[itest]
            if ctlanom and itest == 0:
                label = '_nolabel_'
            ax_row[iprof].plot(mean, pres, linestyle=linestyle[itest], color=linecolor[itest], label=label)
            ax_row[iprof].fill_betweenx(pres, low, high, alpha=0.2, color=linecolor[itest])

        # Row-specific subtitle labels
        if irow == 0:
            row_subtitles = ['(a) LW-CRF', r'(b) $\theta_v$', r'(c) $\theta_e$', '(d) RH', '(e) VMF']
        else:
            row_subtitles = ['(f) LW-CRF', r'(g) $\theta_v$', r'(h) $\theta_e$', '(i) RH', '(j) VMF']

        ax_row[iprof].set_title(row_subtitles[iprof])
        ax_row[iprof].set_xlabel(units[iprof])
        ax_row[iprof].invert_yaxis()
        ax_row[iprof].set_yscale('log')
        ax_row[iprof].set_ylim(1000, 100)
        ax_row[iprof].yaxis.set_major_formatter(ticker.ScalarFormatter())
        ax_row[iprof].yaxis.set_minor_formatter(ticker.ScalarFormatter())
        ax_row[iprof].axvline(x=0, color='k', linewidth=0.5)
        sns.despine(offset=10, ax=ax_row[iprof])

    # WTG profile
    if wtg_prof is not None:
        pltvar_wtg = np.mean(wtg_prof, axis=2)
        for itest in range(ntest):
            mean, low, high = mean_confidence_interval(pltvar_wtg[:,itest])
            ax_row[-1].plot(mean, pres, linestyle=':', linewidth=2, color=linecolor[itest])
            if (itest == 0) or (itest == 1):
                ax_row[-1].fill_betweenx(pres, low, high, alpha=0.2, color=linecolor[itest])

    # Add row label
    # plt.suptitle(title)
    # ax_row[0].text(-0.35, 0.5, r'$\Delta t$='+str(it_max-1)+' hrs',
    #                transform=ax_row[0].transAxes, fontsize=12, fontweight='bold',
    #                va='center', ha='center', rotation=90)
    ax_row[2].annotate(r'$\Delta t$='+str(it_max-1)+' hrs',
                xy=(0.5, 1.11), xycoords='axes fraction',
                ha='center', va='bottom', fontsize=16, fontweight='bold')

plt.savefig(figdir+'profiles_basic_2row.png', dpi=400, facecolor='white',
            bbox_inches='tight')
plt.show()

##### WTG profiles

In [ ]:
# WTG assessment: 4-panel figure
# (upper) Stratiform, (lower) Anvil
# (left) dse_ttend, (right) dse_w_adv

ctlanom = True
conf_shading = True
it_max = 6
# it_max = 30

# Prepare data arrays scaled to 1e-2 J/kg/s, sliced over test time window
dse_ttend_plt = dse_ttend * 1e2
dse_w_adv_plt = dse_w_adv * 1e2
dse_ttend_plt = dse_ttend_plt[..., t0_tests:t0_tests+it_max, :]
dse_w_adv_plt = dse_w_adv_plt[..., t0_tests:t0_tests+it_max, :]

# mean_str indices: 0=Deep, 1=Strat, 2=Anvil, 3=MCS, 4=All
imean_strat = 1  # Strat
imean_anvil = 2  # Anvil

# Panel layout: rows = [Strat, Anvil], cols = [dse_ttend, dse_w_adv]
panel_imean = [imean_strat, imean_strat, imean_anvil, imean_anvil]
panel_data  = [dse_ttend_plt, dse_w_adv_plt, dse_ttend_plt, dse_w_adv_plt]
panel_titles = [
    r'(a) $\partial \overline{s}/\partial t$ — Stratiform',
    r'(b) $\overline{w}\partial \overline{s}/\partial z$ — Stratiform',
    r'(c) $\partial \overline{s}/\partial t$ — Anvil',
    r'(d) $\overline{w}\partial \overline{s}/\partial z$ — Anvil',
]
xlim_all = [(-6, 3)] * 4
units_str = '$10^{-2}$ J/kg/s'

fig, axs = plt.subplots(2, 2, figsize=(8, 9), layout="constrained", dpi=300, sharey=True)
# fig.suptitle('WTG Assessment, TEST-CTL' + r' ($\Delta t$=' + str(it_max - 1) + ' hrs)')
fig.supylabel('Pressure [hPa]')

for ipanel, ax in enumerate(axs.flat):
    imean = panel_imean[ipanel]
    pltvar = panel_data[ipanel][..., imean, :, :]  # (nmem, ntest, nt_sub, nz)
    pltvar_tavg = np.mean(pltvar, axis=2)  # (nmem, ntest, nz)

    # CTL anomaly: subtract CTL for each ensemble member
    if ctlanom:
        pltvar_tavg -= pltvar_tavg[:, 0, :][:, np.newaxis, :]

    for itest in range(ntest):
        mean, low, high = mean_confidence_interval(pltvar_tavg[:, itest])
        label = tests_str[itest]
        if ctlanom and itest == 0:
            label = '_nolabel_'
        ax.plot(mean, pres, linestyle=linestyle[itest], color=linecolor[itest], label=label)
        if conf_shading:
            ax.fill_betweenx(pres, low, high, alpha=0.2, color=linecolor[itest])

    # WTG profile overlay on right-column panels (dse_w_adv)
    if ipanel in [1, 3]:  # right column
        wtg_prof = dse_w_adv_lwcrf_delta[..., imean, t0_tests:t0_tests+it_max, :] * 1e2
        pltvar_wtg = np.mean(wtg_prof, axis=2)
        for itest in range(ntest):
            mean, low, high = mean_confidence_interval(pltvar_wtg[:, itest])
            ax.plot(mean, pres, linestyle=':', linewidth=2, color=linecolor[itest])
            if conf_shading:
            #     # if (itest == 0) or (itest == 1):
                ax.fill_betweenx(pres, low, high, alpha=0.2, color=linecolor[itest])

    ax.set_title(panel_titles[ipanel])
    if ipanel > 1:
        ax.set_xlabel(units_str)
    ax.invert_yaxis()
    ax.set_ylim(1000, 100)
    ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.yaxis.set_minor_formatter(ticker.ScalarFormatter())
    ax.set_xlim(xlim_all[ipanel])
    ax.axvline(x=0, color='k', linewidth=0.5)
    sns.despine(offset=10, ax=ax)

# Add legend to upper-right panel
# axs[0, 1].legend(loc="lower right", frameon=False, fontsize=9)

plt.savefig(figdir + 'profiles_wtg_4panel_strat_anvil.png', dpi=400, facecolor='white',
            bbox_inches='tight')
plt.show()

#### CTL histogram

In [ ]:
# Precip class

# pclass_plot = [1,4,5]#,6] # deep, strat, anvil
pclass_plot = [1,4,5,6] # deep, strat, anvil, MCS
# pclass_plot = [1,2,3,4,5]
nclass_plot = len(pclass_plot)
pclass_names_plot = [pclass_names[i] for i in pclass_plot]
pclass_names_plot = ['Deep', 'Strat', 'Anvil', 'DSA']

pclass_area_plt = np.copy(pclass_area[:,0,pclass_plot,t0_tests+1:t1_tests])
pclass_area_plt = 1e2*np.transpose(pclass_area_plt, (1,0,2))

# Precip efficiency

pclass_plot_pe = [0,1,2,3] # deep, strat, anvil, MCS
pe_plt = (np.copy(1 - (-vmfd/vmfu)))[:,0,pclass_plot_pe,t0_tests+1:t1_tests]
# pe_plt = (np.copy(rain_kgpms / condh_kuk))[:,0,pclass_plot_pe,t0_tests+1:t1_tests]
# pe_plt = (np.copy(rain / condh_vint))[:,0,pclass_plot_pe,t0_tests+1:t1_tests]
pe_plt = np.transpose(pe_plt, (1,0,2))

# Greenhouse enhancement factor (GEF)

# gef = lwcrf_vint / rain # unitless
gef = lwcrf_vint / (lv*rain_kgpms) # unitless
# gef = swcrf_vint / (lv*rain_kgpms) # unitless
# gef = lwcrf_vint # just lwacre
# gef = swcrf_vint # just swacre
# gef = (lv*rain_kgpms) # just rain
# gef = lwcrf_vint / condh_vint # unitless
gef_plt = np.copy(gef[:,0,pclass_plot_pe,t0_tests+1:t1_tests])
gef_plt = np.transpose(gef_plt, (1,0,2))

# Set up vars

# pltvars = [pclass_area_plt, pe_plt, gef_plt]
delta_x2 = 3*3 # grid spacing squared [km2]
pltvars = [pclass_area_plt,
           gef_plt,
        #    np.sqrt(cluster_size*delta_x2)[0,:,pclass_plot,t0_tests+1:t1_tests],
           pe_plt]
cluster_size[0,:,pclass_plot,t0_tests+1:t1_tests]
titles = [
    '(a) Area fraction', 
    # r'$\epsilon = 1 - M_d/M_u$',
    # 'LW-ACRE/$L_vP$',
    # '(b) LWACRE/$LP$',
    '(b) GEF',
    r'(c) $\epsilon$',
    # '(c) Mean Size (Deep)',
    # '(c) $r$',
    ]

units = ['%',
         '',
        #  'km',
         '']

# Create plot

def plot_pclass_pe_hist(pltvars, nclass_plot, pclass_names_plot, titles, units):

    # fig, axs = plt.subplots(3,1, figsize=(4,8), layout="tight", dpi=200)
    fig, axs = plt.subplots(1,3, figsize=(7,3), layout="constrained", dpi=300)

    cmap = ['teal', 'plum', 'darkorange', 'gold', 'cornflowerblue']
    sns.set_palette(cmap)

    for iplt, ax in enumerate(axs):
        pltvar = np.reshape(pltvars[iplt],(nclass_plot,nmem*(nt_test-1)))
        # sns.boxplot([pltvar[ic] for ic in range(nclass_plot)],
        #             ax=ax, width=0.7, showmeans=True, showfliers=True,
        #             meanprops={"marker":"o", "markerfacecolor":"white", 
        #             "markeredgecolor":"black", "markersize":"6"},
        #             notch=True, bootstrap=10000, showcaps=False,
        #             legend='full')
        sns.violinplot([pltvar[ic] for ic in range(nclass_plot)],
                    width=0.7,
                    # palette=colors, alpha=0.8,
                    inner="box", #split=True,
                    # density_norm='area',
                    # gap=0.01,
                    ax=ax, legend='full')
        ax.set_title(titles[iplt])

        ax.set_ylabel(units[iplt])
        if iplt == 0:
            # ax.set_ylim([0, 0.3])
            ax.set_ylim([0, 30])
        # if iplt == 1:
            # ax.set_ylim([0, 1])
        if iplt < 2:
            # remove legend
            ax.get_legend().remove()
        #     # remove x-axis
        #     sns.despine(offset=10, ax=ax, bottom=True)
        #     # remove x-axis ticks
        ax.set_xticks([])
        # else:
        # ax.set_xticklabels(pclass_names_plot)
        sns.despine(offset=10,ax=ax, bottom=True)#, left=True)
        if iplt == 1:
            ax.set_yscale('log')
            ax.set_ylim(0.01, 1)
            # ax.set_ylim(1e-3, 1e2)

    # Customize the legend labels
    handles, labels = axs[2].get_legend_handles_labels()
    axs[2].legend(handles, pclass_names_plot, loc='upper right',
    # axs[3].legend(handles, pclass_names_plot, loc='upper right',
                  bbox_to_anchor=(2.0, 0.75),  # Move the legend outside the plot area
                  borderaxespad=0., frameon=False,)

    plt.savefig(figdir+'ctl_histogram.png',dpi=400, facecolor='white', \
            bbox_inches='tight')#, pad_inches=0.2)
    plt.show()
    return None

plot_pclass_pe_hist(pltvars, nclass_plot, pclass_names_plot, titles, units)

#### Time series, vert integrals

##### 6-panel

In [ ]:
# mean_str_actual = ["Deep", "Strat", "Anvil", 'MCS', 'All']
# pclass_names=["Non-cloud", "Deep", "Cong", "Shallow", "Stratiform", "Anvil", "MCS"]
mainvar_sel = [0,1,2,3]

# subtract_all = True
subtract_all = False

ctlanom = True
# ctlanom = False

ctl_fraction = True
# ctl_fraction = False

conf_shading = True
do_legend = True

delta_x2 = 3*3 # grid spacing squared [km2]

# Variable list
all_vars_imean = np.array([

    # upper-left
    pclass_area[...,6,:]*1e2,

    # upper-center, Mass Flux PE
    (1 - (-vmfd/vmfu))[...,3,:],
    # Kukulies PE
    # (np.copy(rain_kgpms / condh_kuk))[...,3,:],

    # upper-right, M_u
    vmfu[...,3,:],

    # lower-left, Area fraction (Deep)
    # pclass_area[...,1,:]*1e2,
    crh_mid[...,0,:]*1e2,

    # lower-center, Mean size (Deep)
    np.transpose(np.sqrt(cluster_size*delta_x2)[:,:,1,:], (1,0,2)),

    # lower-right, M_d
    vmfd[...,3,:],
    ])

# Plot only sensitivity test times
all_vars_imean = all_vars_imean[...,t0_tests:t1_tests]

# Compute as fraction of CTL to yield percent difference
if ctl_fraction:
    for ivar in range(all_vars_imean.shape[0]):
      ivar_process = all_vars_imean[ivar,...]
      ivar_ctl_mean = np.nanmean(ivar_process[:,0,:], axis=0)[np.newaxis,np.newaxis,:]
      ivar_process /= ivar_ctl_mean
      ivar_process *= 1e2 # convert to %
      all_vars_imean[ivar,...] = ivar_process

subtitles = [
    '(a) Area fraction',
    r'(b) $\epsilon$',# = 1 -M_d/M_u$',
    '(c) $M_u$',
    # '(d) Area fraction (Deep)',
    f'(d) RH ({plevs_int_mid[0]}-{plevs_int_mid[1]} hPa; Deep)',
    '(e) Mean size (Deep)',
    '(f) $M_d$',
    ]
units = [
    '%',
    '-',
    'kg/m/s',
    '%',
    'km',
    'kg/m/s',
    ]
if ctl_fraction:
    units = [' ',' ',' ',' ',' ',' ',
            #  '-',
            #  r'$\Delta/$CTL',
            #  r'$\Delta/$CTL',
            #  r'$\Delta/$CTL',
            #  r'$\Delta/$CTL',
             ]

i_nt = all_vars_imean.shape[-1]
# create figure
fig, axs = plt.subplots(2,3, figsize=(13,5.3), layout="constrained", dpi=300) # row, column
title = storm.capitalize()
# fig.suptitle(title)
if ctl_fraction:
    # fig.supylabel(r'$\Delta/$CTL [%]', x=0.01, fontsize=14)
    fig.supylabel(r'% change from CTL', x=0.01, fontsize=14)
for iprof, ax in enumerate(axs.flat):
    pltvar = np.copy(all_vars_imean[iprof])
    if ctlanom:
        pltvar -= pltvar[:,0,:][:,np.newaxis,:]
    # Plot all sensitivity tests for variable
    for itest in range(ntest):
        tser = pltvar[:,itest,:]
        # Smooth time series
        tser = smooth_tser(tser)
        mean, low, high = mean_confidence_interval(tser)
        ax.plot(mean, linestyle=linestyle[itest], color=linecolor[itest], label=tests_str[itest])
        if conf_shading:
            # if (itest == 0) or (itest == 1):
            xdim = range(0,i_nt)
            ax.fill_between(xdim, high, low, alpha=0.2, color=linecolor[itest])
    ax.set_title(subtitles[iprof])
    ax.set_ylabel(units[iprof])
    # if iprof % 2 == 0:
    if iprof < 3:
        sns.despine(offset=10,ax=ax, bottom=True)
        # Remove tick labels
        ax.set_xticks([])
    else:
        sns.despine(offset=10,ax=ax)
        ax.set_xlabel('Time [hours]')
    if do_legend and iprof == 2:
        ax.legend(frameon=False, fontsize=12, bbox_to_anchor=(1.0, 0.7))

plt.savefig(figdir+'exp_6panel.png',dpi=400, facecolor='white', \
        bbox_inches='tight')#, pad_inches=0.2)
plt.show()

##### PE comparison

In [ ]:
# subtract_all = True
subtract_all = False

ctlanom = True
# ctlanom = False

ctl_fraction = True
# ctl_fraction = False

# Variable list
all_vars_vint = np.array([
    # vmf,
    1 - (-vmfd/vmfu),
    # dqdt_crf_vint,
    # rain,
    # dqdt_condh_vint,
    # condh_vint,
    # rain/condh_vint,
    rain_kgpms/condh_kuk,
    # (-vmfd/vmfu)],
    ])

# Compute as fraction of CTL to yield percent difference
if ctl_fraction:
    for ivar in range(len(all_vars_vint)):
      ivar_process = all_vars_vint[ivar,...]
      ivar_ctl_mean = np.nanmean(ivar_process[:,0,...], axis=0)[np.newaxis,np.newaxis,...]
      ivar_process /= ivar_ctl_mean
      ivar_process *= 1e2 # convert to %
      all_vars_vint[ivar,...] = ivar_process

subtitles = [
    # 'VMF',
    r'(a) $\epsilon_w = 1 - M_d/M_u$',
    # r'$\left< \dot q_{crf} \right>$',
    # 'Rainfall',
    # r'$\left< \dot q_{lh} \right>$',
    # 'C',
    r'(b) $\epsilon_{mp} = P/C$',
    # r'$\epsilon = P/C_{kuk}$',
    ]
units     = [
    # 'kg/m/s',
    '-',
    # 'W/m$^2$',
    # 'W/m$^2$',
    '-',
    # '-',
    ]

if ctl_fraction:
    units = ['% change from CTL', '% change from CTL']

run_tser_driver(all_vars_vint, subtitles, units, ctlanom=ctlanom, subtract_all=subtract_all, do_legend=True,
                figname='pe')

##### RH and conditional instability

In [ ]:
# Two-panel time series of CRH for Deep and Strat means

ctlanom = True
# ctlanom = False

ctl_fraction = True
# ctl_fraction = False

# Variable: CRH (mid-level)
# pltvar_crh = np.copy(crh_mid * 1e2)  # Convert to %
pltvar_crh = np.array([
    crh_mid*1e2,
    crh_mid*1e2,
    the_diff,
    the_diff,
    ])
var_tag = [
    f'RH ({plevs_int_mid[0]}-{plevs_int_mid[1]} hPa)',
    f'RH ({plevs_int_mid[0]}-{plevs_int_mid[1]} hPa)',
    r'$\hat\theta_{e,PBL}-\hat\theta_{e,FT}$',
    r'$\hat\theta_{e,PBL}-\hat\theta_{e,FT}$',
    # 'CRH',
    # r'$\hat\theta_{e,pbl}-\hat\theta_{e,freetrop}$',
]

# Compute as fraction of CTL to yield percent difference
if ctl_fraction:
    ivar_ctl_mean = np.nanmean(pltvar_crh[:,:,0,...], axis=1)[:,np.newaxis,np.newaxis,...]
    pltvar_crh /= ivar_ctl_mean
    pltvar_crh *= 1e2  # convert to %
    # units_str = r'$\Delta/$CTL [%]'
    units_str = '% change from CTL'
else:
    units_str = '%'

# Plot only sensitivity test times
all_vars_plot = pltvar_crh[...,t0_tests:t1_tests]

# Indices for Deep and Strat
imean_deep = 0   # Deep
imean_strat = 1  # Strat
imean_mcs = 3    # MCS
# row_imean = [imean_deep, imean_strat, imean_mcs]
row_imean = [imean_deep, imean_strat, imean_deep, imean_strat]
# row_imean = [imean_mcs, imean_mcs]
row_labels, letters = [], []
for irow, imean in enumerate(row_imean):
    row_labels.append(f'{var_tag[irow]}, {mean_str_names[imean]}')
    letters.append('(' + chr(97+irow) + ')') # (a), (b), etc.

nrows = len(row_imean)
i_nt = all_vars_plot.shape[-1]

fig_x = 6
fig_y = 8
fig, axs = plt.subplots(nrows, 1, figsize=(fig_x, fig_y), layout="constrained", dpi=300)

title = 'CRH'
if ctlanom:
    title += ', TEST-CTL'

for irow in range(nrows):
    imean = row_imean[irow]
    ax = axs[irow]
    pltvar = np.copy(all_vars_plot[irow, ..., imean, :])
    if ctlanom:
        pltvar -= pltvar[:, 0, :][:, np.newaxis, :]
    # Plot all sensitivity tests
    for itest in range(ntest):
        tser = pltvar[:, itest, :]
        # Smooth time series
        tser = smooth_tser(tser)
        mean, low, high = mean_confidence_interval(tser)
        label = tests_str[itest]
        ax.plot(mean, linestyle=linestyle[itest], color=linecolor[itest], label=label)
        xdim = range(0, i_nt)
        ax.fill_between(xdim, high, low, alpha=0.2, color=linecolor[itest])

    # Labels
    # if irow == 0 or irow == 2:
    # ax.set_ylabel(units_str)
    # ax.text(0.02, 0.95, letters[irow] + ' ' + row_labels[irow], transform=ax.transAxes,
    #         verticalalignment='top')#, fontweight='bold')
    ax.set_title(letters[irow] + ' ' + row_labels[irow])

    if irow < nrows - 1:
        sns.despine(offset=10, ax=ax, bottom=True)
        ax.set_xticks([])
    else:
        sns.despine(offset=10, ax=ax)
        ax.set_xlabel('Time [hours]')

    # Legend in top panel only
    # if irow == 0:
    #     ax.legend(loc="lower left", frameon=False, fontsize=9)

fig.supylabel(units_str)

figname = 'crh_deep_strat_' + storm
plt.savefig(figdir + 'time_series_' + figname + '.png', dpi=400, facecolor='white',
            bbox_inches='tight')
plt.show()

##### WTG VADV terms

In [ ]:
# WTG dq time series: 5x2 figure
# (left) Stratiform, (right) Anvil
# Rows: PW_mid, dqdt_vadv, WTG(LWCRF+LH), WTG(LWCRF), WTG(LH)

ctlanom = True
subtract_all = False

# Variable list
all_vars_vint = np.array([
    # pw_mid,
    # crh_mid*1e2,
    dqdt_vadv_mid,
    dqdt_vadv_lwcrf_mid + dqdt_vadv_condh_mid,
    dqdt_vadv_lwcrf_mid,
    dqdt_vadv_condh_mid,
])
# Slice to test times
all_vars_vint = all_vars_vint[..., t0_tests:t1_tests]

# Subtract ALL-MEAN
if subtract_all:
    all_vars_vint -= all_vars_vint[..., -1, :][..., np.newaxis, :]

subtitles_base = [
    # 'PW',
    # 'CRH_mid',
    'VADV',
    'VADV-WTG: LWCRF + LH',
    'VADV-WTG: LWCRF',
    'VADV-WTG: LH',
]
units_list = [
    # 'mm',
    # '%',
    'mm/d',
    'mm/d',
    'mm/d',
    'mm/d',
]
ylim_list = [
    # (-0.7, 0.7),
    (-9, 2),
    # (-9, 2),
    # (-9, 2),
    # (-9, 2),
]*len(subtitles_base)

# mean_str indices: 0=Deep, 1=Strat, 2=Anvil, 3=MCS, 4=All
imean_deep = 0
imean_strat = 1
imean_anvil = 2
# col_labels = ['Deep', 'Stratiform', 'Anvil']
# col_imean  = [imean_deep, imean_strat, imean_anvil]
col_labels = ['Stratiform', 'Anvil']
col_imean  = [imean_strat, imean_anvil]

nvar = all_vars_vint.shape[0]
i_nt = all_vars_vint.shape[-1]
panel_labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l']

fig_x = 12
fig_y = 2.0 * nvar + 1.8
ncols = len(col_labels)
fig, axs = plt.subplots(nvar, ncols, figsize=(fig_x, fig_y), layout="constrained", dpi=300,
                        sharex=True)

# Column subtitles
# for icol, col_title in enumerate(col_labels):
#     axs[0, icol].set_title(col_title, fontsize=14, fontweight='bold', pad=20)

for irow in range(nvar):
    for icol in range(ncols):
        ax = axs[irow, icol]
        imean = col_imean[icol]
        pltvar = np.copy(all_vars_vint[irow, :, :, imean, :])  # (nmem, ntest, nt)

        if ctlanom:
            pltvar -= pltvar[:, 0, :][:, np.newaxis, :]

        for itest in range(ntest):
            tser = pltvar[:, itest, :]
            tser = smooth_tser(tser)
            mean, low, high = mean_confidence_interval(tser)
            label = tests_str[itest]
            if ctlanom and itest == 0:
                label = '_nolabel_'
            ax.plot(mean, linestyle=linestyle[itest], color=linecolor[itest], label=label)
            xdim = range(0, i_nt)
            ax.fill_between(xdim, high, low, alpha=0.2, color=linecolor[itest])

        ipanel = irow * 2 + icol
        if irow == 0:
            # Top row: use annotate to place variable label below the column header
            # ax.annotate(f'({panel_labels[ipanel]}) {subtitles_base[irow]}',
            #             xy=(0.5, 1.0), xycoords='axes fraction',
            #             ha='center', va='bottom', fontsize=10)
            ax.annotate(f'{col_labels[icol]}',
                        xy=(0.5, 1.2), xycoords='axes fraction',
                        ha='center', va='bottom', fontsize=14, fontweight='bold')
            # ax.set_title(f'{col_labels[icol]}\n\n({panel_labels[ipanel]}) {subtitles_base[irow]}')
        # else:
        ax.set_title(f'({panel_labels[ipanel]}) {subtitles_base[irow]}')
        ax.set_ylabel(units_list[irow])
        ax.set_ylim(ylim_list[irow])
        # ax.set_xlim(-2, 20)
        if irow < nvar - 1:
            sns.despine(offset=10, ax=ax, bottom=True)
            ax.xaxis.set_visible(False)
        else:
            sns.despine(offset=10, ax=ax)
            ax.set_xlabel('Time [hours]')

        if icol == 1:
            # Remove y-axis labels on right column
            ax.set_ylabel('')
            # ax.set_yticklabels([])

# Legend on first panel
# axs[0, 0].legend(loc="lower left", frameon=False, fontsize=9)

plt.savefig(figdir + 'tser_wtg_dq_strat_anvil.png', dpi=400, facecolor='white',
            bbox_inches='tight')
plt.show()

##### WTG assessment

In [ ]:
# WTG dq time series: 5x2 figure
# (left) Stratiform, (right) Anvil
# Rows: PW_mid, dqdt_vadv, WTG(LWCRF+LH), WTG(LWCRF), WTG(LH)

ctlanom = False
subtract_all = False

# Variable list
all_vars_vint = np.array([
    dse_ttend_vint_mid,
    dse_vadv_vint_mid,
    np.abs(dse_ttend_vint_mid)/np.abs(dse_vadv_vint_mid),
])
# Slice to test times
all_vars_vint = all_vars_vint[..., t0_tests:t1_tests]

# Subtract ALL-MEAN
if subtract_all:
    all_vars_vint -= all_vars_vint[..., -1, :][..., np.newaxis, :]

subtitles_base = [
    r'$\partial \overline{s}/\partial t$ (TTEND)',
    r'$\overline{w}\partial \overline{s}/\partial z$ (VADV)',
    '|TTEND| / |VADV|',
]

units_list = [
    r'W/m$^2$',
    r'W/m$^2$',
    '-',
]
ylim_list = [
    # (-0.7, 0.7),
    (-9, 2),
    # (-9, 2),
    # (-9, 2),
    # (-9, 2),
]*len(subtitles_base)

# mean_str indices: 0=Deep, 1=Strat, 2=Anvil, 3=MCS, 4=All
imean_deep = 0
imean_strat = 1
imean_anvil = 2
# col_labels = ['Deep', 'Stratiform', 'Anvil']
# col_imean  = [imean_deep, imean_strat, imean_anvil]
col_labels = ['Stratiform', 'Anvil']
col_imean  = [imean_strat, imean_anvil]

nvar = all_vars_vint.shape[0]
i_nt = all_vars_vint.shape[-1]
panel_labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l']

fig_x = 12
fig_y = 2.0 * nvar + 1.8
ncols = len(col_labels)
fig, axs = plt.subplots(nvar, ncols, figsize=(fig_x, fig_y), layout="constrained", dpi=300,
                        sharex=True)

# Column subtitles
# for icol, col_title in enumerate(col_labels):
#     axs[0, icol].set_title(col_title, fontsize=14, fontweight='bold', pad=20)

for irow in range(nvar):
    for icol in range(ncols):
        ax = axs[irow, icol]
        imean = col_imean[icol]
        pltvar = np.copy(all_vars_vint[irow, :, :, imean, :])  # (nmem, ntest, nt)

        if ctlanom:
            pltvar -= pltvar[:, 0, :][:, np.newaxis, :]

        # Recalculate the ratio
        if irow == 2:
            dse_ttend = np.copy(all_vars_vint[0, :, :, imean, :])
            dse_vadv = np.copy(all_vars_vint[1, :, :, imean, :])
            if ctlanom:
                dse_ttend -= dse_ttend[:, 0, :][:, np.newaxis, :]
                dse_vadv -= dse_vadv[:, 0, :][:, np.newaxis, :]
            pltvar = np.abs(dse_ttend) / (np.abs(dse_vadv) + 1e-8)  # Add small value to avoid division by zero

        for itest in range(ntest):
            tser = pltvar[:, itest, :]
            tser = smooth_tser(tser)
            mean, low, high = mean_confidence_interval(tser)
            if irow == 2:
                print(f'{subtitles_base[irow]}, {col_labels[icol]}')
                # it_max = 12
                it_max = 24
                print(np.mean(mean[0:it_max]))
                print()
            label = tests_str[itest]
            if ctlanom and itest == 0:
                label = '_nolabel_'
            ax.plot(mean, linestyle=linestyle[itest], color=linecolor[itest], label=label)
            xdim = range(0, i_nt)
            ax.fill_between(xdim, high, low, alpha=0.2, color=linecolor[itest])

        ipanel = irow * 2 + icol
        if irow == 0:
            # Top row: use annotate to place variable label below the column header
            # ax.annotate(f'({panel_labels[ipanel]}) {subtitles_base[irow]}',
            #             xy=(0.5, 1.0), xycoords='axes fraction',
            #             ha='center', va='bottom', fontsize=10)
            ax.annotate(f'{col_labels[icol]}',
                        xy=(0.5, 1.2), xycoords='axes fraction',
                        ha='center', va='bottom', fontsize=14, fontweight='bold')
            # ax.set_title(f'{col_labels[icol]}\n\n({panel_labels[ipanel]}) {subtitles_base[irow]}')
        # else:
        ax.set_title(f'({panel_labels[ipanel]}) {subtitles_base[irow]}')
        ax.set_ylabel(units_list[irow])
        # ax.set_ylim(ylim_list[irow])
        # ax.set_xlim(-2, 20)
        if irow < nvar - 1:
            sns.despine(offset=10, ax=ax, bottom=True)
            ax.xaxis.set_visible(False)
        else:
            sns.despine(offset=10, ax=ax)
            ax.set_xlabel('Time [hours]')

        if icol == 1:
            # Remove y-axis labels on right column
            ax.set_ylabel('')
            # ax.set_yticklabels([])

# Legend on first panel
# axs[0, 0].legend(loc="lower left", frameon=False, fontsize=9)

plt.savefig(figdir + 'tser_wtg_dse_strat_anvil.png', dpi=400, facecolor='white',
            bbox_inches='tight')
plt.show()